In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from utils.modelling import *

In [ ]:
import os
if os.path.exists("../data/evaluation.parquet"):
    evaluation_df = pd.read_parquet("../data/evaluation.parquet")
else:
    evaluation_df = pd.DataFrame()

## VERSION 1

In [ ]:
insurance_claims = pd.read_parquet("../data/fraud_model_dataset_v1.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns.drop("fraud_reported")
binary_map = {"fraud_reported": {"N": 0, "Y": 1}}

insurance_claims = encode_features(insurance_claims, categorical_columns, binary_map)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "fraud_reported", 0.2, True)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

#### LOGISTIC REGRESSION

In [ ]:
lr = create_pipeline(LogisticRegression(), preprocessor)
lr.fit(x_train, y_train)
lr_results = classification_results(lr, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Logistic Regression", 
#                              "Initial", 0.5, lr_results, True)

In [ ]:
lr_ttc = optimise_threshold(lr, "balanced_accuracy", 5, x_train, y_train)
lr_ttc_results = classification_results(lr_ttc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Logistic Regression", 
#                              "Threshold", lr_ttc.best_threshold_, lr_ttc_results, True)

#### RANDOM FOREST

In [ ]:
rfc = create_pipeline(RandomForestClassifier(class_weight = "balanced", random_state = 123), preprocessor)
rfc.fit(x_train, y_train)
rfc_results = classification_results(rfc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Random Forest", 
#                              "Initial", 0.5, rfc_results, True)

In [ ]:
parameters = {
    "model__n_estimators": [100], 
    "model__criterion": ["entropy"],
    "model__min_samples_split": [2],
    "model__min_samples_leaf": [5], 
    "model__max_depth": [3]
}

rfc_rs = optimise_model(rfc, parameters, "recall", 5, x_train, y_train)
rfc_rs_results = classification_results(rfc_rs, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Random Forest", 
#                              "Optimise", 0.5, rfc_rs_results, True)

In [ ]:
rfc_ttc = optimise_threshold(rfc_rs, "balanced_accuracy", 5, x_train, y_train)
rfc_ttc_results = classification_results(rfc_ttc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Random Forest", 
#                              "Threshold", rfc_ttc.best_threshold_, rfc_ttc_results, True)

#### GRADIENT BOOSTING

In [ ]:
gbc = create_pipeline(GradientBoostingClassifier(random_state = 123), preprocessor)
gbc.fit(x_train, y_train)
gbc_results = classification_results(gbc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Gradient Boosting", 
#                              "Initial", 0.5, gbc_results, True)

In [ ]:
parameters = {
    'model__learning_rate': [0.05],
    'model__n_estimators': [100], 
    'model__max_depth': [2],
    'model__min_samples_split': [2],
    'model__min_samples_leaf': [6]
}

gbc_rs = optimise_model(gbc, parameters, "recall", 5, x_train, y_train)
gbc_rs_results = classification_results(gbc_rs, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Gradient Boosting", 
#                              "Optimise", 0.5, gbc_rs_results, True)

In [ ]:
gbc_ttc = optimise_threshold(gbc_rs, "balanced_accuracy", 5, x_train, y_train)
gbc_ttc_results = classification_results(gbc_ttc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Gradient Boosting", 
#                              "Threshold", gbc_ttc.best_threshold_, gbc_ttc_results, True)

### COMPARISON

In [ ]:
v1_comparison = evaluation_df[evaluation_df["dataset_version"] == "V1"]
v1_comparison.loc[[1, 4, 7], ["model", "stage", "accuracy", "precision_1", "recall_1", "f1_score_1", 
                              "true_positive", "false_negative", "false_positive", "true_negative"]]